# 02 — Data Analysis Report (Deliverable B)

14 numbered tasks. Each cell prints a table/number, draws a chart where useful,
and prints a one- or two-sentence interpretation computed directly from the
data (not hardcoded), so the numbers always match whatever master_train.csv
actually contains.

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from features import attach_price

PROCESSED = os.path.join('..', 'data', 'processed')
ASSETS = os.path.join('..', 'app', 'assets')

master_train = pd.read_csv(os.path.join(PROCESSED, 'master_train.csv'))
price_clean = pd.read_csv(os.path.join(ASSETS, 'price_clean.csv'))
weather_clean = pd.read_csv(os.path.join(ASSETS, 'weather_clean.csv'))

print(master_train.shape, price_clean.shape, weather_clean.shape)

## B1.1 — Revenue per hectare by crop

In [ ]:
revenue_df = attach_price(master_train, price_clean)
b1_1 = revenue_df.groupby('crop_type')['revenue_birr_per_ha'].mean().sort_values(ascending=False)
print(b1_1)

fig, ax = plt.subplots(figsize=(7, 4))
b1_1.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_ylabel('Revenue (birr/ha)')
ax.set_title('B1.1 Average revenue per hectare by crop')
plt.tight_layout()
plt.show()

print(f"{b1_1.idxmax()} has the highest average revenue per hectare at {b1_1.max():.0f} birr/ha, "
      f"about {(b1_1.max() / b1_1.min()):.1f}x the lowest-revenue crop ({b1_1.idxmin()}).")

## B1.2 — Price trend, 2021-2024

In [ ]:
b1_2 = price_clean.groupby(['year', 'crop_type'])['price_birr_per_quintal'].mean().unstack()
print(b1_2)

fig, ax = plt.subplots(figsize=(7, 4))
b1_2.plot(ax=ax, marker='o')
ax.set_ylabel('Price (birr/quintal)')
ax.set_title('B1.2 Crop price trend, 2021-2024')
plt.tight_layout()
plt.show()

change_pct = (b1_2.iloc[-1] - b1_2.iloc[0]) / b1_2.iloc[0] * 100
biggest_mover = change_pct.abs().idxmax()
print(f"{biggest_mover} moved the most from {b1_2.index.min()} to {b1_2.index.max()}: "
      f"{change_pct[biggest_mover]:+.1f} percent.")

## B2.1 — Yield by region

In [ ]:
b2_1 = master_train.groupby('region')['yield_tons_per_ha'].mean().sort_values(ascending=False)
print(b2_1)

fig, ax = plt.subplots(figsize=(7, 4))
b2_1.plot(kind='bar', ax=ax, color='#55A868')
ax.set_ylabel('Mean yield (tons/ha)')
ax.set_title('B2.1 Mean yield by region')
plt.tight_layout()
plt.show()

print(f"{b2_1.idxmax()} has the highest average yield at {b2_1.max():.2f} tons/ha; "
      f"{b2_1.idxmin()} the lowest at {b2_1.min():.2f} tons/ha.")

## B2.2 — Yield by crop

In [ ]:
b2_2 = master_train.groupby('crop_type')['yield_tons_per_ha'].mean().sort_values(ascending=False)
print(b2_2)

fig, ax = plt.subplots(figsize=(7, 4))
b2_2.plot(kind='bar', ax=ax, color='#C44E52')
ax.set_ylabel('Mean yield (tons/ha)')
ax.set_title('B2.2 Mean yield by crop')
plt.tight_layout()
plt.show()

print(f"{b2_2.idxmax()} yields the most on average ({b2_2.max():.2f} tons/ha), "
      f"{b2_2.idxmin()} the least ({b2_2.min():.2f} tons/ha).")

## B2.3 — Region x crop pivot

In [ ]:
b2_3 = master_train.pivot_table(index='region', columns='crop_type', values='yield_tons_per_ha', aggfunc='mean')
print(b2_3.round(2))

best_cell = b2_3.stack().idxmax()
worst_cell = b2_3.stack().idxmin()
print(f"The single best-performing region-crop combination is {best_cell[1]} in {best_cell[0]}, "
      f"at {b2_3.stack().max():.2f} tons/ha. The worst is {worst_cell[1]} in {worst_cell[0]}, "
      f"at {b2_3.stack().min():.2f} tons/ha. See fig04_region_crop_heatmap.png for the full picture.")

best_region_altitude = master_train.loc[master_train['region'] == best_cell[0], 'altitude_m'].mean()
worst_region_altitude = master_train.loc[master_train['region'] == worst_cell[0], 'altitude_m'].mean()
print(f"A plausible agronomic reason: {best_cell[0]} averages {best_region_altitude:.0f}m altitude versus "
      f"{worst_cell[0]}'s {worst_region_altitude:.0f}m -- altitude and the associated climate differences "
      f"across regions are a reasonable explanation for this pattern, though confirming it would need "
      f"agronomic domain input beyond what this dataset alone can prove.")

## B2.4 — Improved seed lift by crop

In [ ]:
b2_4 = master_train.groupby(['crop_type', 'improved_seed_used'])['yield_tons_per_ha'].mean().unstack()
b2_4.columns = ['no_improved_seed', 'improved_seed']
b2_4['lift_tons_per_ha'] = b2_4['improved_seed'] - b2_4['no_improved_seed']
b2_4['lift_pct'] = b2_4['lift_tons_per_ha'] / b2_4['no_improved_seed'] * 100
print(b2_4.round(2))

best = b2_4['lift_pct'].idxmax()
print(f"{best} shows the largest improved-seed lift at {b2_4.loc[best, 'lift_pct']:.1f} percent.")

## B2.5 — Pest/disease gap by crop

In [ ]:
b2_5 = master_train.groupby(['crop_type', 'pest_disease_flag'])['yield_tons_per_ha'].mean().unstack()
b2_5.columns = ['no_pest_disease', 'pest_disease']
b2_5['gap_tons_per_ha'] = b2_5['no_pest_disease'] - b2_5['pest_disease']
print(b2_5.round(2))

worst = b2_5['gap_tons_per_ha'].idxmax()
print(f"{worst} loses the most yield to pest/disease pressure: {b2_5.loc[worst, 'gap_tons_per_ha']:.2f} tons/ha.")

## B3.1 — Fertilizer response bands

In [ ]:
fert_band = pd.qcut(master_train['fertilizer_kg_per_ha'], 5, duplicates='drop')
b3_1 = master_train.groupby(fert_band)['yield_tons_per_ha'].mean()
print(b3_1.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
b3_1.plot(kind='bar', ax=ax, color='#8172B2')
ax.set_ylabel('Mean yield (tons/ha)')
ax.set_title('B3.1 Yield by fertilizer application band')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f"The highest-fertilizer band averages {b3_1.iloc[-1]:.2f} tons/ha versus "
      f"{b3_1.iloc[0]:.2f} tons/ha for the lowest-fertilizer band.")

band_diffs = b3_1.diff().dropna()
if len(band_diffs) >= 2 and band_diffs.iloc[-1] < band_diffs.iloc[0] * 0.5:
    shape = 'flattening out (diminishing returns at higher fertilizer levels)'
elif band_diffs.min() > 0 and band_diffs.max() / max(band_diffs.min(), 0.001) < 2:
    shape = 'roughly linear'
else:
    shape = 'inconsistent / not clearly monotonic'
print(f"Shape of the fertilizer response: {shape}.")

## B3.2 — Altitude bands by crop (flagging small samples)

In [ ]:
altitude_band = pd.cut(master_train['altitude_m'], bins=5)
b3_2 = master_train.groupby([altitude_band, 'crop_type'])['yield_tons_per_ha'].agg(['mean', 'count'])
low_n = b3_2[b3_2['count'] < 30]
print(b3_2.round(2))
print()
print(f"{len(low_n)} altitude-band x crop cells have fewer than 30 plots and should be read cautiously.")

## B3.3 — Planting month effect

In [ ]:
b3_3 = master_train.groupby('planting_month')['yield_tons_per_ha'].mean().sort_values(ascending=False)
print(b3_3.round(2))
print(f"Plots sown in {b3_3.idxmax()} average the highest yield ({b3_3.max():.2f} tons/ha); "
      f"note planting_month is strongly tied to crop_type in this dataset, so this reflects crop choice too.")

## B3.4 — Distance-to-market correlation

In [ ]:
corr_val = master_train['distance_to_market_km'].corr(master_train['yield_tons_per_ha'])

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(master_train['distance_to_market_km'], master_train['yield_tons_per_ha'], alpha=0.2, s=8)
ax.set_xlabel('Distance to market (km)')
ax.set_ylabel('Yield (tons/ha)')
ax.set_title('B3.4 Distance to market vs yield')
plt.tight_layout()
plt.show()

print(f"Correlation between distance to market and yield: {corr_val:.3f} "
      f"({'a weak' if abs(corr_val) < 0.2 else 'a moderate'} relationship).")
print(f"{'Keep' if abs(corr_val) >= 0.05 else 'Consider dropping'} distance_to_market_km as a model feature: "
      f"the correlation is {corr_val:.3f}, which is "
      f"{'large enough to plausibly help' if abs(corr_val) >= 0.05 else 'too weak on its own to expect much benefit'} "
      f"(it stays in the current model pipeline either way, since a weak linear correlation does not rule out "
      f"a non-linear contribution that a tree-based model could still pick up).")

## B4.1 — Yield trend by year

In [ ]:
b4_1 = master_train.groupby('survey_year')['yield_tons_per_ha'].mean()

fig, ax = plt.subplots(figsize=(6, 4))
b4_1.plot(marker='o', ax=ax)
ax.set_ylabel('Mean yield (tons/ha)')
ax.set_title('B4.1 Mean yield by survey year')
plt.tight_layout()
plt.show()

trend = 'risen' if b4_1.iloc[-1] > b4_1.iloc[0] else 'fallen'
print(f"Mean yield has {trend} from {b4_1.iloc[0]:.2f} tons/ha in {b4_1.index.min()} "
      f"to {b4_1.iloc[-1]:.2f} tons/ha in {b4_1.index.max()}.")

year_to_year_change = b4_1.diff().dropna().abs().mean()
yield_spread = master_train['yield_tons_per_ha'].std()
print(f"Average year-to-year change ({year_to_year_change:.3f} tons/ha) compared with the overall "
      f"plot-to-plot yield spread (std = {yield_spread:.3f} tons/ha): the year-to-year change is "
      f"{'small relative to' if year_to_year_change < 0.3 * yield_spread else 'substantial relative to'} the spread, "
      f"so this {'looks like year-to-year noise rather than a real trend' if year_to_year_change < 0.3 * yield_spread else 'may reflect a real multi-year trend, not just noise'}.")

## B4.2 — Weather anomaly by region-year

In [ ]:
b4_2 = master_train.groupby(['region', 'survey_year'])['temp_deviation_from_region_avg'].mean().unstack()
print(b4_2.round(2))

fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(b4_2.values, cmap='coolwarm', aspect='auto')
ax.set_xticks(range(len(b4_2.columns)))
ax.set_xticklabels(b4_2.columns)
ax.set_yticks(range(len(b4_2.index)))
ax.set_yticklabels(b4_2.index)
ax.set_title('B4.2 Temperature deviation from regional norm, by region-year')
plt.colorbar(im, ax=ax, label='deg C deviation')
plt.tight_layout()
plt.show()

hottest = b4_2.stack().idxmax()
print(f"The largest positive temperature anomaly was {hottest[0]} in {hottest[1]}, "
      f"at {b4_2.stack().max():+.2f} deg C above that region's norm.")

anomaly_region, anomaly_year = hottest
region_mask = master_train['region'] == anomaly_region
anomaly_year_yield = master_train.loc[region_mask & (master_train['survey_year'] == anomaly_year), 'yield_tons_per_ha'].mean()
other_years_yield = master_train.loc[region_mask & (master_train['survey_year'] != anomaly_year), 'yield_tons_per_ha'].mean()
print(f"{anomaly_region}'s yield in the anomaly year {anomaly_year} ({anomaly_year_yield:.2f} tons/ha) vs its "
      f"other years ({other_years_yield:.2f} tons/ha): a difference of {anomaly_year_yield - other_years_yield:+.2f} tons/ha.")

by_crop_anomaly = master_train.loc[region_mask & (master_train['survey_year'] == anomaly_year)].groupby('crop_type')['yield_tons_per_ha'].mean()
by_crop_other = master_train.loc[region_mask & (master_train['survey_year'] != anomaly_year)].groupby('crop_type')['yield_tons_per_ha'].mean()
crop_diff = (by_crop_anomaly - by_crop_other).dropna()
print('Per-crop difference in the anomaly year vs other years (tons/ha):')
print(crop_diff.round(2))
same_direction = (crop_diff > 0).all() or (crop_diff < 0).all()
print(f"The effect is "
      f"{'consistent across crops (same direction for all)' if same_direction else 'not consistent -- crops respond differently to the anomaly'}.")

## B4.3 — Plot-reported vs station-derived rainfall agreement

In [ ]:
rain_corr = master_train['rainfall_mm_season'].corr(master_train['season_rainfall_mm_total'])

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(master_train['season_rainfall_mm_total'], master_train['rainfall_mm_season'], alpha=0.2, s=8)
lims = [0, max(master_train['season_rainfall_mm_total'].max(), master_train['rainfall_mm_season'].max())]
ax.plot(lims, lims, 'k--', lw=1)
ax.set_xlabel('Station-derived season rainfall (mm)')
ax.set_ylabel('Plot-reported season rainfall (mm)')
ax.set_title('B4.3 Plot-reported vs station-derived rainfall')
plt.tight_layout()
plt.show()

mad = (master_train['rainfall_mm_season'] - master_train['season_rainfall_mm_total']).abs().mean()
print(f"Correlation between the plot's self-reported season rainfall and the station-derived "
      f"season rainfall: {rain_corr:.3f}. Mean absolute difference: {mad:.1f} mm "
      f"(plot-reported mean {master_train['rainfall_mm_season'].mean():.0f} mm vs "
      f"station-derived mean {master_train['season_rainfall_mm_total'].mean():.0f} mm). "
      f"A correlation well below 1.0 and a non-trivial mean absolute difference suggest the two sources "
      f"disagree meaningfully -- plausibly because the plot-level figure is a farmer's rough estimate "
      f"while the station figure is an aggregated regional average, not a hyper-local measurement.")